# Brain Tumor Classification using MRI Images

Binary classification (`yes` = tumor present, `no` = no tumor) on a small, heavily imbalanced MRI dataset (~259 `yes` vs ~22 `no`).

This notebook covers: dataset inspection, EDA, preprocessing + imbalance handling, stratified splitting, a from-scratch CNN baseline, a transfer-learning model, full evaluation (including why plain accuracy is misleading here), and Grad-CAM explainability.

Reusable functions live in `../src/`; this notebook focuses on orchestration, results, and the reasoning behind each decision (useful for viva).

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Add project root to path so `src` is importable from inside notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src import data_utils, dataset, evaluate, gradcam, models, preprocessing

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)

## 1. Confirm dataset structure

Re-scans `data/` and reports exactly what it found before anything else happens, so a wrong folder layout fails loudly here instead of silently downstream.

In [ ]:
paths, labels = data_utils.load_image_paths_labels(DATA_DIR)
counts = data_utils.summarize_class_counts(labels)

print(f"Found {len(paths)} images under {data_utils.find_dataset_root(DATA_DIR)}")
for cls, info in counts.items():
    print(f"  {cls}: {info['count']} images ({info['pct']}%)")

## 2. Exploratory Data Analysis

### 2.1 Class counts and imbalance

**This dataset is severely imbalanced (~12:1 yes:no).** That single fact drives most of the downstream decisions in this notebook: how we split the data, how we augment it, how we weight the loss, and — critically — which metrics we trust. It is called out explicitly here rather than left implicit, because a plain accuracy number on this dataset is close to meaningless on its own (see Section 6).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
class_names = list(counts.keys())
class_counts = [counts[c]["count"] for c in class_names]
bars = ax.bar(class_names, class_counts, color=["#4C72B0", "#DD8452"])
for bar, c in zip(bars, class_names):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"{counts[c]['count']} ({counts[c]['pct']}%)", ha="center")
ax.set_ylabel("Number of images")
ax.set_title("Class distribution — severe imbalance")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "class_distribution.png"), dpi=150)
plt.show()

n_no, n_yes = counts["no"]["count"], counts["yes"]["count"]
print(f"Imbalance ratio (yes:no) = {n_yes / n_no:.1f} : 1")
print(f"A classifier that always predicts 'yes' would score {100 * n_yes / (n_yes + n_no):.1f}% accuracy "
      "while having 0% recall on 'no' — this is the baseline every real model must clearly beat on "
      "minority-class metrics, not just overall accuracy.")

### 2.2 Image properties: dimensions, format, color mode

MRI datasets scraped from mixed sources commonly mix grayscale and RGB images, and inconsistent dimensions. We check for this explicitly because it determines whether channel conversion (Section 3) is actually necessary, and confirms resizing is required rather than assumed.

In [ ]:
records = data_utils.inspect_dataset(DATA_DIR)

modes = [r.mode for r in records]
formats = [r.format for r in records]
sizes = [(r.width, r.height) for r in records]

from collections import Counter
print("Color modes:", Counter(modes))
print("File formats:", Counter(formats))
print("Unique (width, height) pairs:", len(set(sizes)))
print("Width range:", min(s[0] for s in sizes), "-", max(s[0] for s in sizes))
print("Height range:", min(s[1] for s in sizes), "-", max(s[1] for s in sizes))

grayscale_count = sum(1 for m in modes if m == "L")
rgb_count = sum(1 for m in modes if m in ("RGB", "RGBA"))
print(f"\n{grayscale_count} grayscale images, {rgb_count} RGB(A) images out of {len(records)} total.")
if grayscale_count > 0 and rgb_count > 0:
    print("Confirmed: this dataset mixes grayscale and RGB images -> channel conversion in preprocessing is required, not optional.")

### 2.3 Sample image grid per class

In [ ]:
def show_sample_grid(paths, labels, class_name, n=6):
    class_paths = [p for p, l in zip(paths, labels) if l == class_name][:n]
    fig, axes = plt.subplots(1, len(class_paths), figsize=(2.2 * len(class_paths), 2.6))
    if len(class_paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, class_paths):
        img = Image.open(p)
        ax.imshow(img, cmap="gray" if img.mode == "L" else None)
        ax.set_title(f"{img.mode}, {img.size}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"Sample '{class_name}' images")
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"sample_grid_{class_name}.png"), dpi=150)
    plt.show()

show_sample_grid(paths, labels, "yes")
show_sample_grid(paths, labels, "no")

## 3. Preprocessing strategy

See `src/preprocessing.py` for the full implementation and rationale in comments. Summary of decisions:

- **Resize to 224x224**: matches the input size MobileNetV2/VGG16 were pretrained on — required for transfer learning to actually use the pretrained filters correctly.
- **Grayscale -> 3-channel**: pretrained backbones expect RGB (trained on ImageNet); we stack the single channel 3x rather than modifying the backbone's first layer, which would discard pretrained weights.
- **Normalization**: `[0,1]` scaling for the from-scratch CNN; backbone-specific `preprocess_input` (mean-centering / `[-1,1]` scaling as appropriate) for the transfer-learning model, since pretrained weights expect that exact input distribution.
- **Imbalance handling** (full detail in `src/dataset.py`): stratified splitting + minority-class oversampling with heavier augmentation (train set only) + class weights in the loss. Three separate, complementary mechanisms — not one clever trick — because each addresses a different part of the imbalance problem.

Quick visual sanity check of the augmentation pipelines below.

In [ ]:
sample_no_path = [p for p, l in zip(paths, labels) if l == "no"][0]
sample_img = preprocessing.load_and_resize(sample_no_path)

standard_aug = preprocessing.build_standard_augmentation()
heavy_aug = preprocessing.build_heavy_augmentation()

fig, axes = plt.subplots(2, 5, figsize=(13, 5.5))
for i, ax in enumerate(axes[0]):
    aug_img = standard_aug(sample_img[tf.newaxis, ...], training=True)[0]
    ax.imshow(tf.cast(aug_img, tf.uint8))
    ax.axis("off")
axes[0, 0].set_ylabel("standard (majority)")
for i, ax in enumerate(axes[1]):
    aug_img = heavy_aug(sample_img[tf.newaxis, ...], training=True)[0]
    ax.imshow(tf.cast(tf.clip_by_value(aug_img, 0, 255), tf.uint8))
    ax.axis("off")
fig.suptitle("Standard augmentation (top, majority class) vs Heavy augmentation (bottom, minority class)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "augmentation_comparison.png"), dpi=150)
plt.show()

## 4. Train / Validation / Test split (stratified)

70/15/15, stratified by class, fixed `random_state` for reproducibility. Printed distribution below confirms every split preserves the original ~12:1 ratio — this is what "stratified" is actually verifying, not just asserting.

In [ ]:
splits = dataset.stratified_split(paths, labels, val_size=0.15, test_size=0.15, random_state=SEED)
dataset.print_split_distribution(splits)

In [ ]:
class_weights = dataset.compute_class_weights(splits["train"][1])
print("Class weights (index 0=no, 1=yes):", class_weights)
print("Applied on top of oversampling + heavy augmentation as an additional safeguard, "
      "not a replacement for them.")

## 5a. Model A — Simple CNN baseline (from scratch)

Trained on the `[0,1]`-normalized pipeline (`backbone=None`). Expected to underperform the transfer-learning model — that gap is itself part of the analysis (see `src/models.py` docstring for why).

In [ ]:
BATCH_SIZE = 16
EPOCHS = 40

train_paths, train_labels = splits["train"]
val_paths, val_labels = splits["val"]
test_paths, test_labels = splits["test"]

cnn_train_ds = dataset.build_training_dataset(train_paths, train_labels, batch_size=BATCH_SIZE, backbone=None)
cnn_val_ds = dataset.build_eval_dataset(val_paths, val_labels, batch_size=BATCH_SIZE, backbone=None)
cnn_test_ds = dataset.build_eval_dataset(test_paths, test_labels, batch_size=BATCH_SIZE, backbone=None)

simple_cnn = models.build_simple_cnn()
simple_cnn.summary()

In [ ]:
cnn_callbacks = models.get_callbacks(
    os.path.join(MODELS_DIR, "simple_cnn_best.keras"), monitor="val_loss", patience=8
)

cnn_history = simple_cnn.fit(
    cnn_train_ds,
    validation_data=cnn_val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=cnn_callbacks,
)

## 5b. Model B — Transfer learning (MobileNetV2, frozen base)

Uses the backbone-specific preprocessing pipeline (`backbone="mobilenetv2"`). Base is frozen: with ~280 images we train only the small classification head, letting the pretrained convolutional features do the heavy lifting.

In [ ]:
BACKBONE = "mobilenetv2"

transfer_train_ds = dataset.build_training_dataset(train_paths, train_labels, batch_size=BATCH_SIZE, backbone=BACKBONE)
transfer_val_ds = dataset.build_eval_dataset(val_paths, val_labels, batch_size=BATCH_SIZE, backbone=BACKBONE)
transfer_test_ds = dataset.build_eval_dataset(test_paths, test_labels, batch_size=BATCH_SIZE, backbone=BACKBONE)

transfer_model = models.build_transfer_model(backbone=BACKBONE, freeze_base=True, learning_rate=1e-4)
transfer_model.summary()

In [ ]:
transfer_callbacks = models.get_callbacks(
    os.path.join(MODELS_DIR, "transfer_mobilenetv2_best.keras"), monitor="val_loss", patience=8
)

transfer_history = transfer_model.fit(
    transfer_train_ds,
    validation_data=transfer_val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=transfer_callbacks,
)

## 6. Evaluation

**Why accuracy alone would be misleading here:** with ~92% of images labeled `yes`, a model that predicts `yes` unconditionally scores ~92% accuracy while having zero recall on `no` — the exact class a clinician most needs correctly flagged (missing genuine no-tumor cases, or worse, the inverse in a differently-imbalanced deployment, has real consequences). Every evaluation below reports **per-class** precision/recall/F1 and the confusion matrix specifically so this failure mode can't hide behind a good-looking headline number.

In [ ]:
evaluate.plot_training_curves(cnn_history, os.path.join(RESULTS_DIR, "simple_cnn_curves.png"), "Simple CNN")
evaluate.plot_training_curves(transfer_history, os.path.join(RESULTS_DIR, "transfer_curves.png"), "Transfer (MobileNetV2)")

In [ ]:
cnn_y_true, cnn_y_prob, cnn_y_pred = evaluate.get_predictions(simple_cnn, cnn_test_ds)
cnn_metrics = evaluate.compute_metrics(cnn_y_true, cnn_y_pred, cnn_y_prob)
evaluate.print_metrics(cnn_metrics, "Simple CNN")
evaluate.save_metrics_json(cnn_metrics, os.path.join(RESULTS_DIR, "simple_cnn_metrics.json"))
evaluate.plot_confusion_matrix(cnn_y_true, cnn_y_pred, os.path.join(RESULTS_DIR, "simple_cnn_confusion.png"), "Simple CNN")
evaluate.plot_roc_curve(cnn_y_true, cnn_y_prob, os.path.join(RESULTS_DIR, "simple_cnn_roc.png"), "Simple CNN")

In [ ]:
transfer_y_true, transfer_y_prob, transfer_y_pred = evaluate.get_predictions(transfer_model, transfer_test_ds)
transfer_metrics = evaluate.compute_metrics(transfer_y_true, transfer_y_pred, transfer_y_prob)
evaluate.print_metrics(transfer_metrics, "Transfer (MobileNetV2)")
evaluate.save_metrics_json(transfer_metrics, os.path.join(RESULTS_DIR, "transfer_metrics.json"))
evaluate.plot_confusion_matrix(transfer_y_true, transfer_y_pred, os.path.join(RESULTS_DIR, "transfer_confusion.png"), "Transfer (MobileNetV2)")
evaluate.plot_roc_curve(transfer_y_true, transfer_y_prob, os.path.join(RESULTS_DIR, "transfer_roc.png"), "Transfer (MobileNetV2)")

In [ ]:
print(f"{'Model':<28}{'Accuracy':>10}{'AUC':>8}{'no-Precision':>14}{'no-Recall':>11}{'no-F1':>8}")
for name, m in [("Simple CNN", cnn_metrics), ("Transfer (MobileNetV2)", transfer_metrics)]:
    no_m = m["per_class"]["no"]
    print(f"{name:<28}{m['accuracy']:>10.3f}{m['auc']:>8.3f}{no_m['precision']:>14.3f}{no_m['recall']:>11.3f}{no_m['f1-score']:>8.3f}")

## 7. Explainability — Grad-CAM

Visualizes which regions of a few test images most influenced the transfer-learning model's prediction, to support the "why did the model decide this" discussion in the report/viva.

In [ ]:
n_examples = min(4, len(test_paths))
example_indices = np.linspace(0, len(test_paths) - 1, n_examples, dtype=int)

fig, axes = plt.subplots(1, n_examples, figsize=(3.2 * n_examples, 3.5))
if n_examples == 1:
    axes = [axes]

for ax, idx in zip(axes, example_indices):
    path, true_label = test_paths[idx], test_labels[idx]
    raw_img = preprocessing.load_and_resize(path)
    model_input = preprocessing.normalize_for_backbone(raw_img, BACKBONE)[tf.newaxis, ...]

    heatmap = gradcam.make_gradcam_heatmap(model_input, transfer_model)
    overlay = gradcam.overlay_gradcam(raw_img.numpy(), heatmap)

    pred_prob = transfer_model.predict(model_input, verbose=0)[0, 0]
    pred_label = "yes" if pred_prob >= 0.5 else "no"

    ax.imshow(overlay)
    ax.set_title(f"true={true_label}, pred={pred_label} ({pred_prob:.2f})", fontsize=9)
    ax.axis("off")

fig.suptitle("Grad-CAM — Transfer model (MobileNetV2)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "gradcam_examples.png"), dpi=150)
plt.show()

## Summary / talking points for viva

- The dataset's ~12:1 imbalance is the central constraint of this assignment; every preprocessing and evaluation choice traces back to it.
- Three imbalance techniques were used **together**: stratified splitting (fair splits), oversampling + heavier augmentation of `no` in training only (more effective minority signal), class weights (loss-level safeguard). None of the three alone would be sufficient.
- Accuracy is reported but never trusted alone — per-class precision/recall/F1 and the confusion matrix are what actually show whether the model learned to detect `no`, not just `yes`.
- The from-scratch CNN vs transfer-learning comparison is expected (and, if it appears in your run, should be discussed) to show transfer learning winning, because it starts from ImageNet-pretrained visual features instead of learning them from ~280 images.
- Limitations to state plainly: tiny `no` source set (~22 images), single train/val/test split rather than cross-validation, no external validation set, real overfitting/generalization risk given dataset size vs typical deep-learning data requirements.